# 01 イジング模型の Metropolis 法（Julia）

新潟大学 集中講義「機械学習と物理学」（2026年9月）の計算実習です。講義ノートの第2章・第3章に対応します。

このノートブックでは
1. 2次元イジング模型の配位を Metropolis 法で作り、
2. 温度を変えて磁化 $\langle|m|\rangle$、磁化率 $\chi$、Binder キュムラント $U_4$ を測って臨界温度 $T_c$ を読み、
3. 最後に、ノートブック 02（ニューラルネット）で使う配位のデータを作ってダウンロードします。

**Colab での準備**：メニューの「ランタイム」→「ランタイムのタイプを変更」で **Julia** を選んでから実行してください。
最初のセルはパッケージのインストールで、数分かかります。

In [ ]:
using Pkg
Pkg.add(["Plots", "NPZ"])
using Random, Statistics, Plots, NPZ

## 1. Metropolis 法

講義ノート第3章のコードと同じです。1回の提案では格子点 $(i,j)$ を無作為に選び、スピンを反転したときのエネルギー差
$\Delta E = 2J s_i \sum_{j\in \mathrm{nn}(i)} s_j$ を計算して、確率 $\min(1, e^{-\beta\Delta E})$ で反転を受け入れます。
提案を $N=L^2$ 回行う単位を 1 スイープと呼びます。境界は周期境界です。

In [ ]:
function delta_energy(spins, i, j, J)
    L = size(spins, 1)
    ip = i == L ? 1 : i + 1
    im = i == 1 ? L : i - 1
    jp = j == L ? 1 : j + 1
    jm = j == 1 ? L : j - 1
    nb = spins[ip, j] + spins[im, j] + spins[i, jp] + spins[i, jm]
    return 2 * J * spins[i, j] * nb
end

function metropolis_sweep!(spins, beta, J, rng)
    L = size(spins, 1)
    for _ in 1:L^2
        i = rand(rng, 1:L)
        j = rand(rng, 1:L)
        dE = delta_energy(spins, i, j, J)
        if dE <= 0 || rand(rng) < exp(-beta * dE)
            spins[i, j] = -spins[i, j]
        end
    end
    return spins
end

magnetization(spins) = sum(spins) / length(spins)

function energy(spins, J)
    L = size(spins, 1)
    E = 0.0
    for i in 1:L, j in 1:L
        ip = i == L ? 1 : i + 1
        jp = j == L ? 1 : j + 1
        E -= J * spins[i, j] * (spins[ip, j] + spins[i, jp])   # 右と上の2本だけ数える
    end
    return E
end

function simulate(L, T; J=1.0, ntherm, nmeas, skip, start, rng)
    beta = 1 / T
    spins = start == :cold ? ones(Int, L, L) : rand(rng, (-1, 1), L, L)
    for _ in 1:ntherm
        metropolis_sweep!(spins, beta, J, rng)
    end
    ms = zeros(nmeas)
    es = zeros(nmeas)
    for k in 1:nmeas
        for _ in 1:skip
            metropolis_sweep!(spins, beta, J, rng)
        end
        ms[k] = magnetization(spins)
        es[k] = energy(spins, J) / L^2
    end
    return (; ms, es, spins)
end

## 2. 配位を見る

低温・臨界温度付近・高温で、熱化した配位を画像として表示します。黄と青が $s_i=\pm1$ です。
臨界温度は $T_c = 2/\log(1+\sqrt2) \simeq 2.269J$ です（以下 $J=1$）。

In [ ]:
rng = Xoshiro(2026)
L = 64
plots = []
for T in (1.5, 2.27, 3.5)
    r = simulate(L, T; ntherm=2000, nmeas=1, skip=1, start=:hot, rng=rng)
    push!(plots, heatmap(r.spins, color=:cividis, aspect_ratio=1, axis=nothing,
                         colorbar=false, title="T = $T"))
end
plot(plots..., layout=(1, 3), size=(900, 300))

## 3. 熱化：冷たい開始と熱い開始

全スピン上向き（冷たい開始）とランダム（熱い開始）から走らせて、磁化の時系列を比べます。
両者が同じ値に落ち着いてから測定を始めます（講義ノート第3章）。

In [ ]:
L = 32; T = 1.8
rng = Xoshiro(1)
cold = simulate(L, T; ntherm=0, nmeas=1000, skip=1, start=:cold, rng=rng)
hot  = simulate(L, T; ntherm=0, nmeas=1000, skip=1, start=:hot,  rng=rng)
plot(abs.(cold.ms), label="cold start", xlabel="sweep", ylabel="|m|")
plot!(abs.(hot.ms), label="hot start")

## 4. 温度を変えて測る：$\langle|m|\rangle$、$\chi$、$U_4$

$$\chi = \beta N\left(\langle m^2\rangle - \langle|m|\rangle^2\right),\qquad
U_4 = 1-\frac{\langle m^4\rangle}{3\langle m^2\rangle^2}$$

$U_4$ は $T<T_c$ で $2/3$、$T>T_c$ で $0$ に近づき、異なる $L$ の曲線が $T_c$ 付近で交わります（講義ノート第3章）。
計算を軽くするため、ここでは $L=8, 16$ だけを使います（数分かかります）。

In [ ]:
Ts = 2.0:0.05:2.6
res = Dict()
for L in (8, 16)
    rng = Xoshiro(L)
    absm = Float64[]; chi = Float64[]; U4 = Float64[]
    for T in Ts
        r = simulate(L, T; ntherm=2000, nmeas=4000, skip=2, start=:cold, rng=rng)
        m = r.ms
        push!(absm, mean(abs.(m)))
        push!(chi, L^2 / T * (mean(m .^ 2) - mean(abs.(m))^2))
        push!(U4, 1 - mean(m .^ 4) / (3 * mean(m .^ 2)^2))
    end
    res[L] = (; absm, chi, U4)
end
p1 = plot(xlabel="T", ylabel="<|m|>"); p2 = plot(xlabel="T", ylabel="chi"); p3 = plot(xlabel="T", ylabel="U4")
for L in (8, 16)
    plot!(p1, Ts, res[L].absm, marker=:o, label="L=$L")
    plot!(p2, Ts, res[L].chi,  marker=:o, label="L=$L")
    plot!(p3, Ts, res[L].U4,   marker=:o, label="L=$L")
end
vline!(p3, [2.269], ls=:dash, label="T_c")
plot(p1, p2, p3, layout=(1, 3), size=(1000, 300))

**問**：$U_4$ の2本の曲線が交わる温度を読み取り、厳密な $T_c=2.269$ と比べてみましょう。
測定回数 `nmeas` を増やすと交点はどう変わりますか。

## 5. ノートブック 02 のためのデータを作る

各温度で独立な連鎖を4本ずつ作り、1本から50配位を取り出します（間隔10スイープ）。
$T<T_c$ では冷たい開始、$T>T_c$ では熱い開始にします。
連鎖番号 `chain` も一緒に保存し、ノートブック 02 では連鎖単位で訓練・検証・テストに分けます（講義ノート第4章「データ分割」）。

In [ ]:
L = 16
Ts = collect(1.0:0.1:3.5)
nchain, nconf, ntherm, skip = 4, 50, 2000, 10
n = length(Ts) * nchain * nconf
X = zeros(Int8, n, L, L)          # X[k, :, :] が k 番目の配位
temps = zeros(n); chains = zeros(Int, n)
k = 0
for (it, T) in enumerate(Ts), c in 1:nchain
    rng = Xoshiro(1000 * it + c)
    spins = T < 2.269 ? ones(Int, L, L) : rand(rng, (-1, 1), L, L)
    for _ in 1:ntherm
        metropolis_sweep!(spins, 1 / T, 1.0, rng)
    end
    for _ in 1:nconf
        for _ in 1:skip
            metropolis_sweep!(spins, 1 / T, 1.0, rng)
        end
        global k += 1
        X[k, :, :] .= spins
        temps[k] = T; chains[k] = c - 1
    end
end
npzwrite("ising_L$(L).npz", Dict("configs" => X, "T" => temps, "chain" => chains, "L" => L))
println("saved ising_L$(L).npz: ", size(X), " configurations")

左の「ファイル」パネル（フォルダのアイコン）に `ising_L16.npz` ができます。
右クリック →「ダウンロード」で手元に保存し、ノートブック 02 でアップロードしてください。

（ダウンロードできない場合も、ノートブック 02 は GitHub に置いた同じ形式のデータを読み込めるようになっています。）